# Fed Sentiment and Monetary Policy: Correcting Bias in Regression with AI-Constructed Indices

This notebook estimates the association between Fed communication sentiment and the expected policy path. It illustrates how the function `ols_bca_topic` can be used to correct bias from regressing on AI-constructed indices. The notebook reproduces results from Table 2 of the AEA P&P article.

In [ ]:
# Run this cell first on Google Colab to install the bias-correction package.
# Requires >= 1.4.0, which bundles the Fed sentiment dataset used below.
# The explicit jax pin avoids a circular-import error from Colab's preinstalled jax.
%pip install "ValidMLInference>=1.4.0" "jax>=0.4.34"

## Data

The sample consists of 199 FOMC meetings from 1995-2023. For each meeting, a classifier labels paragraphs in Fed communications as hawkish, dovish, or neutral. The outcome variable is the GSS path factor, which measures expected future policy rates. The control variable is the shadow short rate.

The data contains:
- `hawkish`, `dovish`, `neutral`: raw classifier counts per meeting
- `path_factor`: GSS path factor (outcome)
- `shadow_rate`: shadow short rate (control)
- `w_hawkish`, `w_dovish`, `w_neutral`: estimated true class probabilities (corrected for classifier error)

The confusion matrix `B` is estimated from hand-labeled validation data, and `kappa` is the scaling parameter $\kappa = \sqrt{n} \times \mathbb{E}[C_i^{-1}]$.

In [ ]:
import pandas as pd
import numpy as np
from ValidMLInference import fed_sentiment_data

# Load data bundled with the package
data = fed_sentiment_data()
meetings = data['meetings']
B = data['B']
kappa = data['kappa']

print(f"Sample size: {len(meetings)} FOMC meetings")
print(f"kappa: {kappa:.4f}")
print(f"\nConfusion matrix B:")
print(np.round(B, 4))

In [ ]:
meetings.head()

## Estimation

We compare three approaches:

1. **Naive two-step**: Use raw classifier frequencies as regressors
2. **Two-step with corrected probabilities**: Use estimated true class probabilities $\hat{w}_i$ obtained by inverting the confusion matrix
3. **Bias-corrected**: Apply the additive bias correction via `ols_bca_topic`

In [ ]:
from ValidMLInference import ols, ols_bca_topic

# Outcome and control variables
Y = meetings['path_factor'].values
Q = meetings[['shadow_rate']].values

# Selection matrix: hawkish and dovish (exclude neutral)
S = np.array([[1, 0, 0], [0, 1, 0]], dtype=float)

# Raw classifier frequencies
counts = meetings[['hawkish', 'dovish', 'neutral']].values
P = counts / counts.sum(axis=1, keepdims=True)

# Estimated true class probabilities
W = meetings[['w_hawkish', 'w_dovish', 'w_neutral']].values

### Naive Two-Step (Raw Frequencies)

In [ ]:
mod_naive = ols(Y=Y, X=np.column_stack([P @ S.T, Q]))
print(mod_naive.summary().rename(index={'x1': 'Hawkish', 'x2': 'Dovish', 'x3': 'Shadow Rate'}))

### Two-Step with Corrected Probabilities

In [ ]:
mod_2step = ols(Y=Y, X=np.column_stack([W @ S.T, Q]))
print(mod_2step.summary().rename(index={'x1': 'Hawkish', 'x2': 'Dovish', 'x3': 'Shadow Rate'}))

### Bias-Corrected

In [ ]:
mod_bc = ols_bca_topic(Y=Y, Q=Q, W=W, S=S, B=B, k=kappa)
print(mod_bc.summary().rename(index={'topic_1': 'Hawkish', 'topic_2': 'Dovish', 'Q_1': 'Shadow Rate'}))

## Results Summary

In [ ]:
# Pull estimates by label (not position) so columns stay aligned, and use
# consistent variable names across all three methods.
rename_maps = {
    'Naive':          {'x1': 'Hawkish', 'x2': 'Dovish', 'x3': 'Shadow Rate'},
    'Two-Step':       {'x1': 'Hawkish', 'x2': 'Dovish', 'x3': 'Shadow Rate'},
    'Bias-Corrected': {'topic_1': 'Hawkish', 'topic_2': 'Dovish', 'Q_1': 'Shadow Rate'},
}
summaries = {
    'Naive':          mod_naive.summary().rename(index=rename_maps['Naive']),
    'Two-Step':       mod_2step.summary().rename(index=rename_maps['Two-Step']),
    'Bias-Corrected': mod_bc.summary().rename(index=rename_maps['Bias-Corrected']),
}

variables = ['Hawkish', 'Dovish', 'Shadow Rate']
methods = ['Naive', 'Two-Step', 'Bias-Corrected']

rows = []
for var in variables:
    row = {'Variable': var}
    for m in methods:
        s = summaries[m]
        row[(m, 'Estimate')] = round(s.loc[var, 'Estimate'], 3)
        row[(m, 'CI')] = f"[{s.loc[var, '2.5%']:.3f}, {s.loc[var, '97.5%']:.3f}]"
    rows.append(row)

display_df = pd.DataFrame(rows).set_index('Variable')
display_df.columns = pd.MultiIndex.from_tuples(display_df.columns)
print(display_df.to_string())

## Conclusion

The bias-corrected estimates show that hawkish Fed language has a statistically significant positive effect on the expected policy path (95% CI excludes zero), while the naive and two-step estimates are not significant. The shadow rate coefficient also becomes significant after bias correction. This illustrates how ignoring measurement error in AI-constructed indices can lead to attenuated estimates and incorrect inference.